# Lucrarea de laborator nr. 2 . Compararea grupurilor, corelația și regresia

## Contextul sarcinii

Continuați să lucrați ca analist de date la compania de asigurări **MedGuard** (vezi lucrarea de laborator nr. 1).

După primul raport, product managerul (PM) știe *cine* sunt clienții noștri. Acum vrea să înțeleagă **de ce depind despăgubirile** și dacă acestea pot fi **prognozate**:

> „Nu am nevoie doar de medii. Spune-mi care grupuri de clienți diferă cu adevărat după despăgubiri și care diferențe sunt doar zgomot. Și dă-mi o formulă după care să pot estima despăgubirile așteptate pentru un client nou”.

**Sarcina voastră** este să comparați grupurile de clienți (criteriul t, analiza dispersională), să găsiți legăturile (corelația) și să construiți un model al despăgubirilor (regresia liniară).

În această versiune a lucrării **nu trebuie să calculați nimic manual după formule**: pentru fiecare metodă găsiți și aplicați funcția gata făcută din biblioteci. Principalul este să alegeți corect metoda, să citiți rezultatul și să formulați concluzia.

---

## Setul de date

Același **Medical Cost Personal Dataset** ca în lucrarea de laborator nr. 1. Fișierul `insurance.csv` trebuie să se afle în același dosar cu acest notebook:

- https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv

| Coloană | Descriere |
|---|---|
| `age` | vârsta asiguratului, ani |
| `sex` | sexul: `male` / `female` |
| `bmi` | indicele de masă corporală (kg/m²) |
| `children` | numărul de copii acoperiți de poliță |
| `smoker` | dacă clientul fumează: `yes` / `no` |
| `region` | regiunea: `northeast`, `northwest`, `southeast`, `southwest` |
| `charges` | suma despăgubirilor de asigurare pe an, USD |

---

## Fițuică: funcții utile

| Sarcina | Funcția |
|---|---|
| Interval de încredere după distribuția t și z | `stats.t.interval`, `stats.norm.interval`, `stats.sem` |
| Compararea a două medii | `stats.ttest_ind(a, b, equal_var=False)` |
| Analiza dispersională unifactorială | `stats.f_oneway`, `ols(...)` + `sm.stats.anova_lm` |
| Care anume grupuri diferă | `pairwise_tukeyhsd` (din `statsmodels.stats.multicomp`) |
| Corelația | `df.corr()`, `stats.pearsonr` |
| Regresia liniară | `ols("y ~ x1 + x2", data=df).fit()`, `.summary()`, `.params`, `.pvalues`, `.rsquared_adj`, `.predict()` |
| Grafice | `sns.boxplot`, `sns.heatmap`, `sns.scatterplot`, `sns.regplot`, `sm.qqplot` |

## Reguli de realizare

1. Nivelul de semnificație: $\alpha = 0.05$.
2. Pentru fiecare verificare a unei ipoteze formulați $H_0$ și $H_1$.
3. După fiecare sarcină — **concluzie cu propriile cuvinte**: ce s-a obținut și ce înseamnă aceasta pentru business.
4. Celulele cu `# codul vostru` și `**Concluzie:**` le completați voi.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

df = pd.read_csv("insurance.csv")
print(df.shape)
df.head()

## Sarcina 1. Distribuția t: când datele sunt puține

Departamentul de vânzări a adus date despre doar **15 fumători** dintr-o regiune nouă și vrea să estimeze pe baza lor despăgubirea medie. Să modelăm această situație.

**1.1.** Faceți eșantionul `df[df.smoker == "yes"].charges.sample(15, random_state=3)`. Afișați media lui și eroarea standard (`stats.sem`).

**1.2.** Construiți intervalul de încredere de 95% pentru medie cu două funcții: `stats.norm.interval` și `stats.t.interval` (pentru t indicați `df = n - 1`). Care interval este mai larg? Care dintre ele este corect în această situație și de ce?

In [ ]:
# codul vostru: eșantion din 15 fumători, media și eroarea standard

In [ ]:
# codul vostru: IÎ 95% prin stats.norm.interval și stats.t.interval

**Concluzie:** 

## Sarcina 2. Compararea a două medii: criteriul t Student

Departamentul financiar vrea să introducă un tarif separat pentru fumători. PM cere să se demonstreze că diferența dintre despăgubiri nu este întâmplătoare. Totodată, marketingul afirmă că „bărbații costă mai mult decât femeile”.

**2.1.** Afișați statistica descriptivă a `charges` pe grupuri `smoker` (`df.groupby("smoker").charges.describe()`) și construiți un boxplot.

**2.2.** Formulați $H_0$ și $H_1$ și verificați-le cu funcția `stats.ttest_ind(..., equal_var=False)`. Ce înseamnă valorile obținute `statistic` și `pvalue`?

**2.3.** Verificați afirmația marketingului: diferă oare despăgubirile medii ale bărbaților și femeilor? Apoi repetați același test **doar printre nefumători**. Priviți proporția fumătorilor printre bărbați și femei (`pd.crosstab(df.sex, df.smoker, normalize="index")`) și explicați ce s-a întâmplat (amintiți-vă din prelegere despre „a treia variabilă”).

In [ ]:
# codul vostru: statistica descriptivă pe smoker și boxplot

In [ ]:
# codul vostru: testul t fumători vs nefumători

In [ ]:
# codul vostru: bărbați vs femei (toți și doar nefumătorii), proporția fumătorilor pe sexe

**Concluzie:** 

## Sarcina 3. Analiza dispersională unifactorială: diferă oare regiunile?

Conducătorul regiunii `southeast` cere mărirea bugetului: „la noi despăgubirile sunt mai mari decât la toți ceilalți”. Regiuni sunt patru, de aceea avem nevoie de **ANOVA**.

**3.1.** Afișați mediile `charges` pe regiuni și construiți boxplot-ul `charges` după `region`.

**3.2.** Formulați $H_0$ și $H_1$. Efectuați ANOVA în două moduri: `stats.f_oneway` și tabelul `sm.stats.anova_lm(ols("charges ~ C(region)", data=df).fit())`. Coincid F și p? Care este concluzia?

**3.3.** ANOVA nu spune **care anume** regiuni diferă. Aplicați criteriul Tukey `pairwise_tukeyhsd(df.charges, df.region)`. Care perechi de regiuni diferă semnificativ (`reject = True`)?

**3.4.** Efectuați ANOVA pentru `bmi` pe regiuni (`stats.f_oneway`) și comparați mediile `bmi`. Cum ar putea aceasta să explice diferențele dintre despăgubiri?

In [ ]:
# codul vostru: mediile pe regiuni și boxplot

In [ ]:
# codul vostru: stats.f_oneway și anova_lm

In [ ]:
# codul vostru: criteriul Tukey

In [ ]:
# codul vostru: ANOVA pentru bmi pe regiuni

**Concluzie:** 

## Sarcina 4. Corelația: ce este legat de despăgubiri?

**4.1.** Construiți matricea de corelații Pearson pentru `age`, `bmi`, `children`, `charges` (`df[[...]].corr()`) și desenați-o ca hartă termică (`sns.heatmap(..., annot=True)`). Care variabilă este cel mai puternic legată de `charges`?

**4.2.** Cu funcția `stats.pearsonr` obțineți coeficientul de corelație dintre `age` și `charges` și valoarea lui p. Calculați coeficientul de determinație $R^2 = r^2$ și explicați sensul lui.

**4.3.** Construiți diagrama de dispersie `age` → `charges`, colorând punctele după `smoker` (`sns.scatterplot(..., hue="smoker")`). Ce se vede? Calculați `stats.pearsonr` între `age` și `charges` **doar pentru nefumători** și comparați cu cel general. De ce trebuie să ne uităm la grafic înainte de a calcula corelația?

In [ ]:
# codul vostru: matricea de corelații și heatmap

In [ ]:
# codul vostru: pearsonr pentru age și charges, R^2

In [ ]:
# codul vostru: scatter după smoker; pearsonr doar pentru nefumători

**Concluzie:** 

## Sarcina 5. Regresia cu o singură variabilă independentă

PM cere „o formulă”: cât adaugă în medie la despăgubiri fiecare an de vârstă?

**5.1.** Construiți modelul `ols("charges ~ age", data=df).fit()` și afișați `summary()`. Găsiți în tabel $b_0$ (Intercept), $b_1$ (age), valorile lor p și $R^2$. Scrieți ecuația modelului și interpretați $b_1$.

**5.2.** Desenați datele cu dreapta de regresie (`sns.regplot(x="age", y="charges", data=df)`).

**5.3.** Cu metoda `.predict()` găsiți prognoza despăgubirii pentru un client de 50 de ani.

**5.4.** Verificați condițiile de aplicare a regresiei: construiți graficul reziduurilor (`model.resid`) în funcție de `age` și QQ-plot-ul reziduurilor (`sm.qqplot(model.resid, line="s")`). Sunt îndeplinite normalitatea reziduurilor și homoscedasticitatea?

In [ ]:
# codul vostru: modelul ols și summary

In [ ]:
# codul vostru: regplot

In [ ]:
# codul vostru: prognoza pentru 50 de ani

In [ ]:
# codul vostru: graficul reziduurilor și QQ-plot

**Concluzie:** 

## Sarcina 6. Regresia multiplă și alegerea celui mai bun model

**6.1.** Construiți mai multe modele și comparați-le după $R^2$ **ajustat** (`.rsquared_adj`):
- `charges ~ age`
- `charges ~ age + bmi`
- `charges ~ age + bmi + smoker`
- `charges ~ age + bmi + smoker + children`
- `charges ~ age + bmi + smoker + children + sex + region`

Adunați rezultatul într-un singur tabel. De ce trebuie comparat anume $R^2$ ajustat?

**6.2.** Alegeți cel mai bun model din punctul de vedere „calitate / simplitate”. Afișați `summary()` al acestuia și interpretați coeficienții, mai ales cel pentru `smoker` (nu uitați: „în condiții egale”).

**6.3.** În modelul complet priviți valorile p (`.pvalues`) pentru `sex` și `region`. Comparați cu concluziile sarcinilor 2 și 3.

**6.4.** Folosind modelul ales, prognozați despăgubirea pentru doi clienți de 50 de ani cu `bmi = 30`: unul fumător și unul nefumător.

In [ ]:
# codul vostru: compararea modelelor după R^2 și R^2 ajustat

In [ ]:
# codul vostru: summary al modelului ales

In [ ]:
# codul vostru: valorile p în modelul complet

In [ ]:
# codul vostru: prognoza pentru clientul fumător și cel nefumător